# Classifier Comparison on the Combined CIC-IDS2018 + CIC-DDoS2019 Dataset

Trains several different classifier algorithms on the **exact same dataset, features, and train/test split** used for `train_experimental_models_2019.ipynb`'s Variant 1 (single-flow XGBoost) - same 28 CIC-IDS2018 offset files + 3 full days + 18 CIC-DDoS2019 attack-type files, same 8 `BASE_FEATURES`, same per-day chronological split.

**Why this notebook exists:** Variant 1/2 both flagged a shortcut warning - `Min Pkt Size` carries 51-59% of feature importance, because 15 of 17 CIC-DDoS2019 attack types are reflection/amplification attacks (DNS/NTP/SSDP/LDAP/SNMP/NetBIOS/MSSQL/Portmap) whose defining trait IS a large forward payload (median 229-1472 bytes) vs benign's near-zero. Live testing against `simulate_attacks.py`'s raw ICMP/UDP/SYN/port-scan floods (which don't share that signature) showed near-zero recall for all 3 existing variants.

The open question this notebook answers: **is that a limitation of XGBoost specifically, or of the data/feature set itself?** If a fundamentally different algorithm (linear vs. bagged trees vs. boosted trees) shows the same shortcut reliance and the same live-test blind spot, that's strong evidence the problem is the data/features, not the model architecture - matching the recommendation that ensembling/voting across similarly-blind models won't help either.

**Scope:** single-flow features only (no temporal/sequence context) - isolates "does the algorithm matter" from "does temporal context help" (already covered separately by Variant 2/3). Outputs to `models/classifier_comparison/` - a comparison/exploration directory, not wired into the live app.

Run `python fetch_ddos2019_sample.py` first if `data/ddos2019_sample/combined_sample.csv` doesn't exist yet, and make sure `data/cicids2018/` has the 28 `_off0`-`_off3` files (via `fetch_cicids2018_multiday_v2.py`).

In [1]:
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARTIAL_DIR = PROJECT_ROOT / "data" / "cicids2018"
FULL_DIR = PROJECT_ROOT / "data" / "cicids2018_full"
DDOS2019_SAMPLE = PROJECT_ROOT / "data" / "ddos2019_sample" / "combined_sample.csv"
OUT_DIR = PROJECT_ROOT / "models" / "classifier_comparison"

PARTIAL_DAYS = [
    "02-14-2018_off0.csv", "02-14-2018_off1.csv", "02-14-2018_off2.csv", "02-14-2018_off3.csv",
    "02-15-2018_off0.csv", "02-15-2018_off1.csv", "02-15-2018_off2.csv", "02-15-2018_off3.csv",
    "02-16-2018_off0.csv", "02-16-2018_off1.csv", "02-16-2018_off2.csv", "02-16-2018_off3.csv",
    "02-20-2018_off0.csv", "02-20-2018_off1.csv", "02-20-2018_off2.csv", "02-20-2018_off3.csv",
    "02-21-2018_off0.csv", "02-21-2018_off1.csv", "02-21-2018_off2.csv", "02-21-2018_off3.csv",
    "02-22-2018_off0.csv", "02-22-2018_off1.csv", "02-22-2018_off2.csv", "02-22-2018_off3.csv",
    "02-23-2018_off0.csv", "02-23-2018_off1.csv", "02-23-2018_off2.csv", "02-23-2018_off3.csv",
]
FULL_DAYS = ["02-28-2018.csv", "03-01-2018.csv", "03-02-2018.csv"]

BASE_FEATURES = ["Flow Duration", "Tot Fwd Pkts", "TotLen Fwd Pkts",
                 "Flow Byts/s", "Flow Pkts/s", "Avg Pkt Size", "Min Pkt Size", "Protocol"]

print(f"comparing classifiers on {len(PARTIAL_DAYS)} 2018 offset files + {len(FULL_DAYS)} full 2018 days + CIC-DDoS2019")

comparing classifiers on 28 2018 offset files + 3 full 2018 days + CIC-DDoS2019


## Load CIC-IDS2018 days (identical to train_experimental_models_2019.ipynb)

In [2]:
def load_one_day(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    df = df[df["Label"] != "Label"].reset_index(drop=True)

    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
    df = df.dropna(subset=["Timestamp"])
    df["Binary_Label"] = df["Label"].apply(lambda x: 0 if str(x).strip().lower() == "benign" else 1)

    for col in ["TotLen Fwd Pkts", "Tot Fwd Pkts", "Fwd Pkt Len Min", "Flow Duration",
                "Flow Byts/s", "Flow Pkts/s", "Protocol", "Dst Port"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Avg Pkt Size"] = df["TotLen Fwd Pkts"] / df["Tot Fwd Pkts"].replace(0, 1)
    df["Min Pkt Size"] = df["Fwd Pkt Len Min"]

    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=BASE_FEATURES + ["Dst Port"])
    df["day"] = path.stem
    return df.sort_values("Timestamp").reset_index(drop=True)


def per_day_split(day_df):
    attacks = day_df.loc[day_df["Binary_Label"] == 1, "Timestamp"]
    if len(attacks) < 20:
        cutoff = day_df["Timestamp"].quantile(0.8)
    else:
        cutoff = attacks.quantile(0.7)
    train_mask = (day_df["Timestamp"] < cutoff).values
    return train_mask, ~train_mask


print("loading 7 CIC-IDS2018 days x 4 offset windows each (28 files) + 3 full days ...")
frames = [load_one_day(PARTIAL_DIR / n) for n in PARTIAL_DAYS]
frames += [load_one_day(FULL_DIR / n) for n in FULL_DAYS]
print(f"  {len(frames)} 2018 day-frames loaded")

loading 7 CIC-IDS2018 days x 4 offset windows each (28 files) + 3 full days ...
  31 2018 day-frames loaded


## Load CIC-DDoS2019 sample (identical to train_experimental_models_2019.ipynb)

In [3]:
if not DDOS2019_SAMPLE.exists():
    raise FileNotFoundError(
        f"{DDOS2019_SAMPLE} not found - run `python fetch_ddos2019_sample.py` from the project root first."
    )

df2019 = pd.read_csv(DDOS2019_SAMPLE, low_memory=False)
df2019["Timestamp"] = pd.to_datetime(df2019["Timestamp"])
ddos2019_frames = [g.sort_values("Timestamp").reset_index(drop=True) for _, g in df2019.groupby("day")]
frames += ddos2019_frames
print(f"  {len(ddos2019_frames)} 2019 day-frames loaded ({len(df2019)} rows total)")
print(f"combined: {len(frames)} day-frames")

  18 2019 day-frames loaded (727078 rows total)
combined: 49 day-frames


## Combine + per-day chronological split (identical to train_experimental_models_2019.ipynb)

In [4]:
train_parts, test_parts = [], []
for day_df in frames:
    tr, te = per_day_split(day_df)
    day_df = day_df.assign(is_test=te)
    train_parts.append(day_df[tr])
    test_parts.append(day_df[te])

df = pd.concat(train_parts + test_parts, ignore_index=True)
train_mask = np.concatenate([np.ones(len(p), dtype=bool) for p in train_parts] +
                             [np.zeros(len(p), dtype=bool) for p in test_parts])
test_mask = ~train_mask

print(f"combined: {len(df)} rows  train={train_mask.sum()}  test={test_mask.sum()}")
print(f"train attack ratio: {df.loc[train_mask,'Binary_Label'].mean():.3f}")
print(f"test attack ratio:  {df.loc[test_mask,'Binary_Label'].mean():.3f}")

y = df["Binary_Label"].values
X = df[BASE_FEATURES]
attack_labels_test = df.loc[test_mask, "Label"]
attack_types = sorted(df.loc[df.Binary_Label == 1, "Label"].unique())
print(f"attack types ({len(attack_types)}): {attack_types}")

combined: 3821984 rows  train=2919482  test=902502
train attack ratio: 0.363
test attack ratio:  0.509
attack types (31): ['Bot', 'Brute Force -Web', 'Brute Force -XSS', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DDoS attacks-LOIC-HTTP', 'DoS attacks-GoldenEye', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'DoS attacks-Slowloris', 'DrDoS_DNS', 'DrDoS_LDAP', 'DrDoS_MSSQL', 'DrDoS_NTP', 'DrDoS_NetBIOS', 'DrDoS_SNMP', 'DrDoS_SSDP', 'DrDoS_UDP', 'FTP-BruteForce', 'Infilteration', 'LDAP', 'MSSQL', 'NetBIOS', 'Portmap', 'SQL Injection', 'SSH-Bruteforce', 'Syn', 'TFTP', 'UDP', 'UDP-lag', 'WebDDoS']


## Helpers

In [5]:
def tune_threshold(y_true, probs):
    """F1-optimal threshold, not the naive 0.5 default - same convention as the other notebooks."""
    return float(max(np.linspace(0.05, 0.95, 91), key=lambda v: f1_score(y_true, probs >= v, zero_division=0)))


def eval_at_threshold(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
    }


def get_importance(fitted_estimator, feature_names, X_sample=None, y_sample=None):
    """Works for tree-based models (.feature_importances_) and linear models
    (.coef_, taken as absolute value since sign doesn't matter for "how much
    does this feature drive the decision"). HistGradientBoostingClassifier
    exposes neither (a real sklearn quirk, not a bug here) - falls back to
    permutation importance on a subsample in that case. Normalized to sum
    to 1 so the same 0.5 shortcut threshold applies across model types."""
    if hasattr(fitted_estimator, "feature_importances_"):
        raw = np.asarray(fitted_estimator.feature_importances_, dtype=float)
    elif hasattr(fitted_estimator, "coef_"):
        raw = np.abs(np.asarray(fitted_estimator.coef_, dtype=float)).ravel()
    elif X_sample is not None and y_sample is not None:
        from sklearn.inspection import permutation_importance
        result = permutation_importance(fitted_estimator, X_sample, y_sample, n_repeats=5, random_state=42, n_jobs=-1)
        raw = np.clip(result.importances_mean, a_min=0, a_max=None)
    else:
        return None, None
    total = raw.sum()
    raw = raw / total if total > 0 else raw
    report = sorted(zip(feature_names, [float(v) for v in raw]), key=lambda x: -x[1])
    shortcut_warning = report[0][1] > 0.5
    return report, shortcut_warning

## Train 4 classifiers on the identical feature set + split

- **XGBoost** - the baseline, identical hyperparameters to Variant 1 in `train_experimental_models_2019.ipynb`, included here for direct comparison.
- **Random Forest** - bagged (not boosted) trees. Different bias/variance tradeoff than XGBoost; if it shows the same shortcut reliance, that rules out boosting-specific overfitting as the cause.
- **HistGradientBoosting** - sklearn's native fast GBM (histogram-binned, similar family to LightGBM). A second boosted-tree implementation with different regularization defaults than XGBoost.
- **Logistic Regression** - the only non-tree model here. Trees can carve arbitrarily sharp splits on one dominant feature; a linear model distributes weight differently across correlated features, so if it *still* leans overwhelmingly on `Min Pkt Size`, that's strong evidence the signal itself is the issue, not how tree-based models exploit it.

In [6]:
def make_models():
    return {
        "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, eval_metric="logloss", random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42),
        "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=200, random_state=42),
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]),
    }


results = {}
fitted_models = {}

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# HistGradientBoosting has no native .feature_importances_/.coef_, so its
# importance needs permutation_importance - capped to a subsample since
# that repeatedly re-predicts the full set n_repeats times per feature.
rng = np.random.default_rng(42)
perm_sample_idx = rng.choice(len(X_test), size=min(20000, len(X_test)), replace=False)
X_perm_sample = X_test.iloc[perm_sample_idx]
y_perm_sample = y_test[perm_sample_idx]

for name, model in make_models().items():
    print(f"training {name} ...")
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    thr = tune_threshold(y_test, probs)
    metrics = eval_at_threshold(y_test, probs, thr)

    # For the Pipeline (Logistic Regression), pull the underlying fitted
    # classifier out for importance - .coef_ lives on the "clf" step, not
    # the pipeline object itself. Permutation importance still needs the
    # full pipeline (it has to go through the scaler too), so pass model
    # itself as the estimator when falling back to that path.
    has_native = hasattr(model, "feature_importances_") or hasattr(model, "coef_")
    if isinstance(model, Pipeline):
        estimator_for_importance = model.named_steps["clf"]
        has_native = hasattr(estimator_for_importance, "feature_importances_") or hasattr(estimator_for_importance, "coef_")
    else:
        estimator_for_importance = model

    if has_native:
        importance, shortcut_warning = get_importance(estimator_for_importance, BASE_FEATURES)
    else:
        importance, shortcut_warning = get_importance(model, BASE_FEATURES, X_perm_sample, y_perm_sample)

    fitted_models[name] = model
    results[name] = {
        "metrics": metrics,
        "probs": probs,
        "preds": (probs >= thr).astype(int),
        "importance": importance,
        "shortcut_warning": shortcut_warning,
    }
    print(f"  {metrics}")
    print(f"  top feature: {importance[0]}  shortcut_warning={shortcut_warning}")
    print()

training XGBoost ...
  {'accuracy': 0.9209907568071871, 'precision': 0.9393212949317832, 'recall': 0.9031426774244802, 'f1': 0.920876784561064, 'threshold': 0.26999999999999996}
  top feature: ('Min Pkt Size', 0.5889974148720888)  shortcut_warning=True

training Random Forest ...
  {'accuracy': 0.9157442310377152, 'precision': 0.9301806855819548, 'recall': 0.9022154798465118, 'f1': 0.9159846865211555, 'threshold': 0.24999999999999994}
  top feature: ('Avg Pkt Size', 0.20040846467409906)  shortcut_warning=False

training HistGradientBoosting ...
  {'accuracy': 0.9197619506660373, 'precision': 0.9382932751107528, 'recall': 0.9016865854534454, 'f1': 0.9196257823486387, 'threshold': 0.25999999999999995}


TypeError: 'NoneType' object is not subscriptable

## Per-attack-type recall breakdown

The aggregate metrics above can look strong while still missing entire attack *shapes* - exactly what happened with the existing Variant 1/2 (92-94% offline accuracy, ~0% live recall on non-reflection floods). Breaking recall down by the original attack `Label` shows whether a different algorithm generalizes across attack shapes any better, or shares the same blind spots (e.g. `Syn`/`TFTP`, the only two CIC-DDoS2019 types that *don't* have the large-payload reflection signature).

In [ ]:
per_type_rows = []
for label in attack_types:
    mask = (attack_labels_test == label).values
    n = int(mask.sum())
    if n == 0:
        continue
    row = {"attack_type": label, "n": n}
    for name, r in results.items():
        row[name] = float(r["preds"][mask].mean())  # fraction flagged attack = recall for this subset
    per_type_rows.append(row)

per_type_df = pd.DataFrame(per_type_rows).sort_values("n", ascending=False).reset_index(drop=True)
pd.set_option("display.width", 160)
print(per_type_df.to_string(index=False, float_format=lambda v: f"{v:.3f}" if isinstance(v, float) else str(v)))

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay, roc_auc_score

model_names = list(results.keys())

fig, axes = plt.subplots(1, len(model_names), figsize=(4.2 * len(model_names), 4.5))
for ax, name in zip(axes, model_names):
    ConfusionMatrixDisplay.from_predictions(
        y_test, results[name]["preds"], display_labels=["Benign", "Attack"], cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name in model_names:
    auc = roc_auc_score(y_test, results[name]["probs"])
    RocCurveDisplay.from_predictions(y_test, results[name]["probs"], name=f"{name} (AUC={auc:.3f})", ax=ax)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Chance")
ax.set_title("ROC Curves")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name in model_names:
    PrecisionRecallDisplay.from_predictions(y_test, results[name]["probs"], name=name, ax=ax)
ax.set_title("Precision-Recall Curves")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
metric_names = ["accuracy", "precision", "recall", "f1"]
x = np.arange(len(metric_names))
width = 0.8 / len(model_names)

fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(model_names):
    values = [results[name]["metrics"][k] for k in metric_names]
    bars = ax.bar(x + i * width - 0.4 + width / 2, values, width, label=name)
    for bar, v in zip(bars, values):
        ax.annotate(f"{v:.2f}", (bar.get_x() + bar.get_width() / 2, v), ha="center", va="bottom", fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.05)
ax.set_title("Metrics comparison across classifiers (identical data + features)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(model_names), figsize=(4.5 * len(model_names), 5))
for ax, name in zip(axes, model_names):
    imp = results[name]["importance"]
    names = [n for n, _ in imp][::-1]
    values = [v for _, v in imp][::-1]
    colors = ["#d62728" if v > 0.5 else "#1f77b4" for v in values]
    ax.barh(names, values, color=colors)
    ax.axvline(0.5, color="grey", linestyle="--", linewidth=1, label="shortcut threshold (0.5)")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("normalized importance")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Save artifacts + provenance

Saved to `models/classifier_comparison/` - kept separate from `models/experimental/` and `models/experimental_2019/` (the artifacts the live app actually loads), since this notebook is for comparison, not deployment. Promote manually if one of these turns out to generalize better.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, model in fitted_models.items():
    safe_name = name.lower().replace(" ", "_")
    with open(OUT_DIR / f"model_{safe_name}.pkl", "wb") as f:
        pickle.dump(model, f)
    with open(OUT_DIR / f"threshold_{safe_name}.pkl", "wb") as f:
        pickle.dump(results[name]["metrics"]["threshold"], f)
    print(f"  saved {safe_name}")

with open(OUT_DIR / "features.pkl", "wb") as f:
    pickle.dump(BASE_FEATURES, f)

provenance = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "dataset": ("CIC-IDS2018, 10 days (7 partial samples + 3 full days) "
                "+ CIC-DDoS2019, 18 attack-type files (own benign only, no cross-dataset blending) "
                "- identical to train_experimental_models_2019.ipynb Variant 1"),
    "total_rows": int(len(df)),
    "train_rows": int(train_mask.sum()),
    "test_rows": int(test_mask.sum()),
    "attack_types": attack_types,
    "features": BASE_FEATURES,
    "note": ("Classifier-algorithm comparison on identical single-flow data/features to test whether "
             "the Min Pkt Size shortcut reliance (flagged in Variant 1/2) is algorithm-specific or "
             "a property of the data/feature set itself."),
    "models": {
        name: {
            "metrics": results[name]["metrics"],
            "feature_importances": results[name]["importance"],
            "shortcut_warning": results[name]["shortcut_warning"],
        }
        for name in model_names
    },
    "per_attack_type_recall": per_type_df.to_dict(orient="records"),
}

with open(OUT_DIR / "provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2, default=str)
print(f"saved {OUT_DIR / 'provenance.json'}")

print("\n=== SUMMARY (F1-optimal thresholds) ===")
print(f"{'Model':<22}{'Threshold':<11}{'Accuracy':<10}{'Precision':<10}{'Recall':<10}{'F1':<10}{'Shortcut?':<10}")
for name in model_names:
    m = results[name]["metrics"]
    print(f"{name:<22}{m['threshold']:<11.3f}{m['accuracy']:<10.4f}{m['precision']:<10.4f}{m['recall']:<10.4f}{m['f1']:<10.4f}{str(results[name]['shortcut_warning']):<10}")